# Kapitel 3 — Ein Modell anpassen · Lösung

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch03_corrige.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Jede Antwort beginnt mit der Überlegung auf Papier und zeigt dann ihre Entsprechung in Python.

**Verwendeter Verlust**: der mittlere quadratische Fehler (Gleichung 3.2) auf den Wohnungen A bis F, sofern nicht anders angegeben.

**Inhalt.** Daten · Die Rechnungen der Lektionen 3.1 bis 3.6 · Aufgaben 3.1 bis 3.6

*Die Zelle von Lektion 3.4 und die von Aufgabe 3.4 f) führen Hunderttausende von Schritten aus: Rechnen Sie jeweils mit einigen Sekunden.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farben des Buchs: eine Farbe = eine Rolle (beobachtet, vorhergesagt, Abweichung, Gerade des Kollegen)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Daten

Die Kennung D0.1 ist die des Buchs und der CSV-Datei.

Variablen- und Spaltennamen im Code bleiben französisch: `logements` = Wohnungen, `logement` = Wohnung, `surface_m2` = Fläche in m², `prix_kEUR` = Preis in k€, `ventes_nouvelles` = neue Verkäufe, `vente` = Verkauf, `eqm` / `eam` = MSE / MAE, `perte` = Verlust, `derivee` = Ableitung, `pas` = Schritt(e), `ecart` = Abweichung, `moyenne` = Mittelwert, `mediane` = Median.

In [ ]:
# Daten von Kapitel 3, aus der CSV-Datei des Buchs übernommen (nicht ändern)

# D0.1 — die sechs Wohnungen aus der Einleitung, für die Anpassung verwendet
logements = pd.DataFrame({
    'logement':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'surface_m2': [32, 45, 54, 70, 85, 98],
    'prix_kEUR':  [118, 172, 185, 245, 279, 335],
})
x = logements['surface_m2'].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)

# D3.1 — vier neue Verkäufe, nur zur Bewertung verwendet (Lektion 3.6)
ventes_nouvelles = pd.DataFrame({
    'vente':      ['V1', 'V2', 'V3', 'V4'],
    'surface_m2': [38, 58, 76, 92],
    'prix_kEUR':  [132, 215, 254, 322],
})
xv = ventes_nouvelles['surface_m2'].to_numpy(dtype=float)
yv = ventes_nouvelles['prix_kEUR'].to_numpy(dtype=float)

# Wohnung G (Einleitung): Rechenbeispiel, nie für die Anpassung verwendet
x_G, y_G = 62, 209

## Die Rechnungen der Lektionen, in Python

Diese Zellen wiederholen die Rechnungen der Lektionen 3.1 bis 3.6.

In [ ]:
# Lektion 3.1 — zwei Geraden, zwei Fehlermaße (Gleichungen 3.1 und 3.2)
def eam(a, b):
    return np.mean(np.abs(y - (a * x + b)))

def eqm(a, b):
    return np.mean((y - (a * x + b)) ** 2)

for nom, a, b in [("Einleitung", 3, 30), ('Kollege', 3.3, 14)]:
    print(f'{nom:10s} MAE = {eam(a, b):.2f} k€   MSE = {eqm(a, b):.2f} k€²   RMSE = {np.sqrt(eqm(a, b)):.2f} k€')

In [ ]:
# Lektion 3.2 — die Verlustkurve, b fest auf 30 k€ (Gleichung 3.3)
pentes = np.round(np.arange(2.5, 3.45, 0.1), 1)
print(pd.Series([round(eqm(a, 30), 2) for a in pentes], index=pentes))
A = np.linspace(2.4, 3.6, 241)
plt.figure(figsize=(6, 3))
plt.plot(A, [eqm(a, 30) for a in A], color=BLEU)
plt.scatter(pentes, [eqm(a, 30) for a in pentes], color='black', zorder=3)
plt.xlabel('Steigung a (k€ pro m²)'); plt.ylabel('Verlust L(a) (k€²)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lektion 3.3 — Sekanten, Ableitung (Gleichung 3.4) und zwei Aktualisierungen (Gleichung 3.5)
def derivee_a(a, b=30):
    e = y - (a * x + b)
    return -2 / len(x) * np.sum(x * e)

for h in (0.1, 0.01, 0.001):
    print(f'Sekante h = {h}: {(eqm(2.9 + h, 30) - eqm(2.9, 30)) / h:.1f}')
print('exakte Ableitung bei 2,9:', round(derivee_a(2.9), 1))

a, eta = 2.5, 0.00006
for etape in range(3):
    print(f'Schritt {etape}: a = {a}, L = {eqm(a, 30):.2f}, L\'(a) = {derivee_a(a):.2f}')
    a = round(a - eta * derivee_a(a), 3)    # auf Tausendstel gerundet, wie im Buch

In [ ]:
# Lektion 3.4 — der Gradient (Gleichung 3.6) und ein Schritt für beide Koeffizienten (Gleichung 3.7)
def gradient(a, b):
    e = y - (a * x + b)
    return np.array([-2 / len(x) * np.sum(x * e), -2 / len(x) * np.sum(e)])

g = gradient(2.5, 30)
print('Gradient bei (2,5 ; 30):', g.round(2))
print('nach einem Schritt:', (np.array([2.5, 30]) - 0.00006 * g).round(4))

a, b = 2.5, 30.0
for k in range(1, 100001):
    a, b = np.array([a, b]) - 0.00006 * gradient(a, b)
    if k in (10, 1000, 100000):
        print(f'nach {k} Schritten: a = {a:.4f}, b = {b:.4f}, L = {eqm(a, b):.2f}')
a_mc, b_mc = np.polyfit(x, y, 1)
print(f'Talboden: a = {a_mc:.3f}, b = {b_mc:.2f}, L = {eqm(a_mc, b_mc):.2f}')
print(f'G, Rechenbeispiel: {a_mc * x_G + b_mc:.0f} k€ vorhergesagt für {y_G} k€ beobachtet')

In [ ]:
# Lektion 3.5 — zwei Abstiege mit denselben Startvorhersagen (Gleichung 3.8)
mu, sigma = x.mean(), x.std()          # std() teilt durch n, wie in Kapitel 1
z = (x - mu) / sigma
print(f'mu = {mu:.0f} m², sigma = {sigma:.3f} m², z = {z.round(2)}')

def descente_m2(eta, nb_pas, a=2.5, b=30.0):
    for _ in range(nb_pas):
        a, b = np.array([a, b]) - eta * gradient(a, b)
    return a, b

def descente_std(eta, nb_pas, a=2.5, b=30.0):
    w, c = a * sigma, b + a * mu        # Gleichung 2.7: gleiche Vorhersagen am Start
    for _ in range(nb_pas):
        e = y - (w * z + c)
        w, c = w + eta * 2 / len(x) * np.sum(z * e), c + eta * 2 / len(x) * np.sum(e)
    return w / sigma, c - w / sigma * mu  # Gleichung 3.8: zurück zu (a ; b)

print('Schritte | m², η = 0,000 06           | standardisiert, η = 0,3')
a, b = 2.5, 30.0
fait = 0
for k in (0, 1, 5, 10, 1000, 100000):
    a, b = descente_m2(0.00006, k - fait, a, b); fait = k
    a_s, b_s = descente_std(0.3, min(k, 1000))
    print(f'{k:>8} | a = {a:.3f} b = {b:6.2f} L = {eqm(a, b):8.2f} | '
          f'a = {a_s:.3f} b = {b_s:6.2f} L = {eqm(a_s, b_s):8.2f}')

# Grenz-Lernraten: darüber divergiert der Abstieg (etwa 0,000 22 in m², 1 standardisiert)
for eta in (0.000216, 0.000217):
    print(f'm², η = {eta}: Verlust nach 400 Schritten = {eqm(*descente_m2(eta, 400)):.4g} k€²')
for eta in (0.99, 1.01):
    print(f'standardisiert, η = {eta}: Verlust nach 400 Schritten = {eqm(*descente_std(eta, 400)):.4g} k€²')

In [ ]:
# Lektion 3.6 — Ausgleichsgerade der kleinsten Quadrate (Gleichung 3.9) und R² (Gleichung 3.10)
cov = np.mean((x - x.mean()) * (y - y.mean()))
a_mc = cov / x.var()
b_mc = y.mean() - a_mc * x.mean()
print(f'cov = {cov:.1f} m²·k€, var = {x.var():.2f} (m²)², a = {a_mc:.3f} k€/m², b = {b_mc:.2f} k€')

def r2_lecon(observes, predits, reference):
    """R² = 1 − MSE der Regel / MSE der konstanten Vorhersage « reference »."""
    return 1 - np.mean((observes - predits) ** 2) / np.mean((observes - reference) ** 2)

y_moy = y.mean()                      # mittlerer Preis von A bis F: die Referenz für R² auf A bis F
print(f'A bis F: MSE konstant = {np.mean((y - y_moy) ** 2):.2f}, MSE Gerade = {eqm(a_mc, b_mc):.2f}, '
      f'R² = {r2_lecon(y, a_mc * x + b_mc, y_moy):.3f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.3f}')

# Die drei Geraden, dasselbe Maß, auf A bis F, dann auf V1 bis V4 (Lektion 3.6)
droites = {'moindres carrés': (a_mc, b_mc), 'ouverture': (3, 30), 'collègue': (3.3, 14)}
for nom, (a, b) in droites.items():
    e_af, e_v = y - (a * x + b), yv - (a * xv + b)
    print(f'{nom:16s} A bis F: MSE = {np.mean(e_af ** 2):6.2f}  MAE = {np.mean(np.abs(e_af)):5.2f}   |   '
          f'V1 bis V4: MSE = {np.mean(e_v ** 2):6.2f}  MAE = {np.mean(np.abs(e_v)):5.2f}   Abweichungen {e_v.round(2)}')
yv_bis = np.array([132, 215, 254, 312.0])   # falls V4 für 312 k€ verkauft worden wäre
for nom in ('moindres carrés', 'collègue'):
    a, b = droites[nom]
    print(f'V4 zu 312 k€: {nom} MSE = {np.mean((yv_bis - (a * xv + b)) ** 2):.2f}')
print(f'G, Rechenbeispiel: {a_mc * x_G + b_mc:.2f} k€ vorhergesagt, Abweichung {y_G - (a_mc * x_G + b_mc):.2f} k€')

## Aufgabe 3.1 — Ein Diagramm ablesen: zwei Maße, zwei Minima

*Daten D0.1. Begriffe: MAE und MSE (Lektion 3.1), Verlustkurve (Lektion 3.2), Median (Lektion 1.3).* Die Steigung ist auf 3 k€/m² festgelegt. Beantworten Sie zuerst die Fragen a) bis e), indem Sie die Abbildung im Buch ablesen.

### a) bis e) Lösung

**a)** Der MSE ist bei etwa $b \approx 30$ k€ am kleinsten und beträgt dort etwa 57 k€². Die Rechnung in f) ergibt genau $b = 30{,}33$ und 57,22 k€².

**b)** Die MAE-Kurve besteht aus Geradenstücken. Um ihr Minimum herum bildet sie ein waagerechtes **Plateau**, von $b = 24$ bis $b = 35$ k€, in einer Höhe von etwa 7,3 k€. Alle diese Werte von $b$ ergeben denselben MAE.

**c)** Für $b = 30$ liest man einen MSE von etwa 57 k€² und einen MAE von etwa 7,3 k€. Die Tabelle der Abweichungen in Lektion 3.1 ergibt 57,33 k€² und 7,33 k€ für die Gerade der Einleitung: Die Ablesung ist stimmig.

**d)** Mit $b = 25$ betragen die Abweichungen $r_i - 25$: −3, 12, −2, 10, −1 und 16 k€. Ihre Beträge ergeben zusammen 44, wie für $b = 30$: Der MAE beträgt weiterhin 44 / 6 ≈ 7,33 k€. Nach dem MAE sind die beiden Geraden **gleichwertig**. Ihre Quadrate ergeben 9 + 144 + 4 + 100 + 1 + 256 = 514, also einen MSE von 514 / 6 ≈ 85,67 k€² gegenüber 57,33: Nach dem MSE ist die Gerade des Kollegen **schlechter**. Das Urteil hängt vom Maß ab, wie in Lektion 3.1.

**e)** Auf dem Plateau ist die Steigung der Kurve 0. Ein Gradientenabstieg, der dort ankommt, bleibt stehen, wo er gerade ist: Er kann nicht zwischen 24 und 35 k€ wählen. An den Knickstellen ändert sich die Steigung abrupt, und die Ableitung existiert nicht. Die MSE-Kurve ist eine glatte Parabel: Sie hat einen einzigen tiefsten Punkt, und ihre Steigung, proportional zum Abstand zu diesem Punkt, zeigt bei jedem Schritt, in welche Richtung und wie weit es gehen soll.

### f) Im Notebook

In [ ]:
def eqm_b(b):
    return np.mean((y - (3 * x + b)) ** 2)

def eam_b(b):
    return np.mean(np.abs(y - (3 * x + b)))

In [ ]:
# Überprüfung, dann Darstellung der beiden Kurven
if eqm_b(30) is None or eam_b(30) is None:
    print('Ergänzen Sie eqm_b und eam_b.')
else:
    print('eqm_b und eam_b:', 'richtig' if np.isclose(eqm_b(30), 344 / 6) and np.isclose(eam_b(30), 44 / 6) else 'bitte prüfen')
    B = np.linspace(15, 45, 601)
    fig, (h, bas) = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)
    h.plot(B, [eqm_b(b) for b in B], color=BLEU); h.set_ylabel('MSE (k€²)'); h.grid(alpha=0.4)
    bas.plot(B, [eam_b(b) for b in B], color=ORANGE); bas.set_ylabel('MAE (k€)'); bas.grid(alpha=0.4)
    bas.set_xlabel('Achsenabschnitt b (k€)')
    plt.show()
    print('Knickstellen der MAE-Kurve (r_i sortiert):', np.sort(y - 3 * x))

In [ ]:
r_tries = np.sort(y - 3 * x)                    # 22, 23, 24, 35, 37, 41
mediane_r = (r_tries[2] + r_tries[3]) / 2       # n = 6 gerade: Ränge 3 und 4 → 29,5
moyenne_r = np.mean(y - 3 * x)                  # 182 / 6 ≈ 30,33

In [ ]:
# Überprüfung
for nom, la_votre, vrai in [('Median', mediane_r, 29.5), ('Mittelwert', moyenne_r, 182 / 6)]:
    print(f'{nom} der r:', 'noch nicht ausgefüllt' if la_votre is None else ('richtig' if np.isclose(la_votre, vrai) else 'bitte prüfen'))
if mediane_r is not None and eam_b(30) is not None:
    print(f'MAE am Median: {eam_b(mediane_r):.2f} k€ (auf dem Plateau); '
          f'MSE am Mittelwert: {eqm_b(moyenne_r):.2f} k€² (Minimum)')

### Lösung zu f)

Die Knickstellen liegen bei 22, 23, 24, 35, 37 und 41 k€, den sechs Werten $r_i$. Das Plateau reicht vom dritten bis zum vierten sortierten Wert. Der Median der $r_i$, $(24 + 35)/2 = 29{,}5$ k€, liegt darauf. Das Minimum des MSE wird beim Mittelwert der $r_i$ erreicht, 30,33 k€. Hier zeigt sich wieder Kapitel 1: Der Mittelwert gehört zum MSE, der Median zum MAE. Der Median ist robust gegenüber Extremwerten, der Mittelwert nicht. Ebenso bestraft der MSE große Fehler stärker als der MAE.

## Aufgabe 3.2 — Zwei Schritte Gradientenabstieg

*Daten D0.1. Begriffe: mittlerer quadratischer Fehler (Lektion 3.1), Ableitung und Aktualisierung (Lektion 3.3).* Die Steigung ist auf 3 k€/m² festgelegt. Gesucht ist der Achsenabschnitt $b$. Bearbeiten Sie zuerst die Fragen a) bis d) auf Papier.

### a) Vorbereiten

In [ ]:
r = y - 3 * x
r

In [ ]:
# Überprüfung
if r is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('r:', 'richtig' if np.allclose(r, [22, 37, 23, 35, 24, 41]) else 'bitte prüfen')

### a) bis c) Lösung auf Papier

**a)** $r = (22 ; 37 ; 23 ; 35 ; 24 ; 41)$ k€. Zum Beispiel für A: 118 − 3 × 32 = 118 − 96 = 22. Die Abweichung ist dann $e_i = y_i - 3x_i - b = r_i - b$.

**b) und c)** Mit $\eta = 0{,}3$ lautet die Aktualisierung $b - 0{,}3 \times \left(-\frac{2}{6}\sum e_i\right) = b + 0{,}1 \sum e_i$.

| Schritt | $b$ (k€) | Abweichungen $r_i - b$ (k€) | $\sum e_i$ | $L(b)$ (k€²) | $L'(b)$ | Schrittweite |
|:--:|--:|:--|--:|--:|--:|--:|
| 0 | 20 | 2 ; 17 ; 3 ; 15 ; 4 ; 21 | 62 | 984 / 6 = 164 | −20,67 | +6,2 |
| 1 | 26,2 | −4,2 ; 10,8 ; −3,2 ; 8,8 ; −2,2 ; 14,8 | 24,8 | 445,84 / 6 ≈ 74,31 | −8,27 | +2,48 |
| 2 | 28,68 | −6,68 ; 8,32 ; −5,68 ; 6,32 ; −4,68 ; 12,32 | 9,92 | ≈ 59,96 | −3,31 | |

Einzelheiten zu Schritt 0: Die Quadrate sind 4, 289, 9, 225, 16 und 441, mit der Summe 984. Die Ableitung beträgt $-\frac{2}{6} \times 62 \approx -20{,}67$. Die Schrittweite beträgt $0{,}3 \times 20{,}67 = 6{,}2$, und $b_1 = 20 + 6{,}2 = 26{,}2$.

In [ ]:
L_b0 = 164
derivee_b0 = -20.67
b1 = 26.2
L_b1 = 74.31
b2 = 28.68
L_b2 = 59.96

In [ ]:
# Überprüfung (unabhängige Berechnung)
def _L(b): return np.mean((y - 3 * x - b) ** 2)
def _d(b): return -2 / 6 * np.sum(y - 3 * x - b)
_b1 = 20 - 0.3 * _d(20); _b2 = _b1 - 0.3 * _d(_b1)
attendus = {'L_b0': (L_b0, _L(20)), 'derivee_b0': (derivee_b0, _d(20)), 'b1': (b1, _b1),
            'L_b1': (L_b1, _L(_b1)), 'b2': (b2, _b2), 'L_b2': (L_b2, _L(_b2))}
for nom, (la_votre, vrai) in attendus.items():
    etat = 'noch nicht ausgefüllt' if la_votre is None else ('richtig' if abs(la_votre - vrai) < 0.01 else 'bitte prüfen')
    print(f'{nom}: {etat}')

### d) Interpretieren

Der Verlust sinkt bei jedem Schritt: 164, dann 74,31, dann 59,96 k€². Die Schritte werden kürzer: 6,2, dann 2,48. Mit $\eta = 0{,}3$ verringert jeder Schritt den Abstand zum Minimum um 60 %. Denn $b_{\text{nouveau}} = b + 0{,}1 \times 6\,(\bar{r} - b)$, also wird $\bar{r} - b$ mit 0,4 multipliziert. Der Abstieg läuft auf $\bar{r} = 182/6 \approx 30{,}33$ k€ zu, den **Mittelwert der $r_i$**. Im Minimum ist der Verlust gleich der Varianz der $r_i$, etwa 57,22 k€², also $7{,}56^2$: Das ist die Standardabweichung der Abweichungen der Geraden aus der Einleitung (Aufgabe 1.6).

### e) Im Notebook

In [ ]:
def perte(b):
    """MSE der Geraden y = 3x + b auf A bis F (Gleichung 3.2)."""
    e = y - (3 * x + b)
    return np.mean(e ** 2)

def derivee(b):
    """L'(b) = −(2/n) Σ e_i (Gleichung 3.6)."""
    e = y - (3 * x + b)
    return -2 / len(x) * np.sum(e)

Die beiden Funktionen übernehmen die Gleichungen 3.2 und 3.6 wörtlich, mit einer auf 3 festgelegten Steigung.

In [ ]:
def descente(b0, eta, nb_pas):
    """Gibt die Liste der b und die Liste der Verluste zurück, von Schritt 0 bis Schritt nb_pas."""
    bs, pertes = [b0], [perte(b0)]
    for _ in range(nb_pas):
        bs.append(bs[-1] - eta * derivee(bs[-1]))   # Gleichung 3.5
        pertes.append(perte(bs[-1]))
    return bs, pertes

if perte(20) is None or derivee(20) is None:
    print('Ergänzen Sie perte und derivee.')
else:
    bs, pertes = descente(20, 0.3, 2)
    print('zwei Schritte:', np.round(bs, 3), np.round(pertes, 2), '→',
          'richtig' if np.allclose(bs, [20, 26.2, 28.68]) else 'bitte prüfen')
    bs, pertes = descente(20, 0.3, 20)
    plt.figure(figsize=(6, 2.8))
    plt.plot(pertes, marker='o', color=BLEU)
    plt.xlabel('Schritt'); plt.ylabel('Verlust (k€²)'); plt.grid(alpha=0.4)
    plt.show()
    print('b nach 20 Schritten:', round(bs[-1], 3), '  Mittelwert der r:', round(np.mean(y - 3 * x), 3))

### f) Vorhersagen, dann überprüfen: $\eta = 1{,}2$

**Vorhersage.** Mit $\eta = 1{,}2$ multipliziert die Aktualisierung den Abstand zum Minimum mit $1 - 2\eta = -1{,}4$. Jeder Schritt springt über das Minimum hinweg und landet auf der anderen Seite weiter weg: $b$ pendelt um 30,33 und entfernt sich dabei immer mehr, und der Verlust steigt. Der Abstieg **divergiert**: 20 → 44,8 → 10,08 → 58,69 → −9,36… Eine zu große Lernrate bremst den Abstieg nicht: Sie lässt ihn scheitern.

In [ ]:
if perte(20) is not None and derivee(20) is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
    for ax, eta in zip(axes, (0.3, 1.2)):
        bs, pertes = descente(20, eta, 6)
        ax.plot(bs, marker='o', color=ORANGE if eta > 1 else BLEU)
        ax.axhline(np.mean(y - 3 * x), color='black', linewidth=0.8, linestyle='--')
        ax.set_title(f'η = {eta}', fontsize=10); ax.set_xlabel('Schritt'); ax.set_ylabel('b (k€)')
        print(f'η = {eta} :', np.round(bs, 2))
    plt.tight_layout(); plt.show()

### g) Rechenbeispiel zu G

In [ ]:
prix_G = 3 * x_G + 28.68     # = 186 + 28,68 = 214,68 k€
ecart_G = y_G - prix_G       # = 209 − 214,68 = −5,68 k€

In [ ]:
# Überprüfung
if prix_G is None or ecart_G is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('G:', 'richtig' if np.isclose(prix_G, 214.68) and np.isclose(ecart_G, -5.68) else 'bitte prüfen')

### Lösung zu g)

Vorhergesagter Preis: 3 × 62 + 28,68 = 186 + 28,68 = **214,68 k€**. Vorzeichenbehaftete Abweichung: 209 − 214,68 = **−5,68 k€**. G wurde nicht zur Anpassung von $b$ verwendet. Diese Rechnung zeigt, wie man die nach zwei Schritten erhaltene Regel anwendet, misst aber nicht ihre Qualität auf neuen Verkäufen: Eine einzige Wohnung würde das nicht erlauben, und Kapitel 6 behandelt diese Frage.

## Aufgabe 3.3 — Was bedeutet „99 %“?

*Daten D0.1 und D3.1. Begriffe: Bestimmtheitsmaß, Anpassung und Bewertung (Lektion 3.6).* Bearbeiten Sie zuerst die Fragen a) bis f) auf Papier.

### a) bis f) Lösung

**a)** Mit Gleichung 3.10 und den MSE-Werten aus Lektion 3.1: Gerade der Einleitung, $R^2 = 1 - 57{,}33 / 5\,205{,}22 \approx 0{,}989$; Gerade des Kollegen, $R^2 = 1 - 65{,}11 / 5\,205{,}22 \approx 0{,}987$. Die Referenz ist die konstante Vorhersage gleich dem mittleren Preis von A bis F, 222,33 k€, und beide MSE-Werte werden auf A bis F berechnet.

**b)** Der Nenner, der MSE der konstanten Vorhersage auf A bis F, ist für alle Geraden gleich. Das $R^2$ ist also umso größer, je kleiner der MSE ist. Die Ausgleichsgerade der kleinsten Quadrate ist aber per Definition diejenige, deren MSE auf A bis F am kleinsten ist. Keine Gerade kann auf diesen sechs Verkäufen ein besseres $R^2$ haben.

**c)** Vorhergesagte Preise: 3 × 38 + 30 = 144, 3 × 58 + 30 = 204, 3 × 76 + 30 = 258 und 3 × 92 + 30 = 306 k€. Abweichungen: 132 − 144 = −12, 215 − 204 = +11, 254 − 258 = −4 und 322 − 306 = +16 k€. Quadrate: 144, 121, 16 und 256, mit der Summe 537. MSE: 537 / 4 = 134,25 k€².

**d)** Auf A bis F: kleinste Quadrate (45,36), Einleitung (57,33), Kollege (65,11). Auf V1 bis V4: Kollege (70,73), kleinste Quadrate (93,29), Einleitung (134,25). Die Umkehrung beweist nicht, dass die Gerade des Kollegen besser ist. Die drei Geraden liegen nah beieinander, und vier Verkäufe sind sehr wenig: Wäre V4 für 312 k€ statt 322 verkauft worden, läge die Ausgleichsgerade wieder vorn (61,11 gegenüber 73,73 k€²). Sie zeigt nur, dass sich die bei der Anpassung erhaltene Rangfolge nicht unverändert auf neue Verkäufe überträgt.

**e)** Zum Beispiel: „Auf den sechs Verkäufen, mit denen wir unsere Gerade gezeichnet haben, verringert sie den mittleren quadratischen Fehler der konstanten Vorhersage gleich ihrem mittleren Preis, 222,33 k€, um 98,9 %. Auf vier neuen Verkäufen beträgt ihr mittlerer quadratischer Fehler 134,25 k€², also etwa 11,6 k€.“ Das Wort „erklärt“ sollte man vermeiden: Das $R^2$ vergleicht zwei Fehler, es misst keine Ursache.

**f)** Das hieße, auf V1 bis V4 zu wiederholen, was die Anpassung auf A bis F getan hat. Sobald man diese Verkäufe zum Auswählen verwendet, sind sie für die gewählte Gerade nicht mehr neu: Ihr MSE auf V1 bis V4 wird seinerseits zu einem Auswahlfehler, und zwar einem optimistischen. Um einen ehrlichen Fehler anzugeben, bräuchte man weitere Verkäufe, die weder zum Anpassen noch zum Auswählen verwendet wurden. Kapitel 6 wird diese Aufteilung organisieren.

### g) Im Notebook

In [ ]:
def eqm_ventes(observes, predits):
    """observes und predits: NumPy-Arrays gleicher Länge."""
    return np.mean((observes - predits) ** 2)

In [ ]:
# Überprüfung: die Rangfolge aus Lektion 3.6, dann R² auf A bis F (Referenz: mittlerer Preis von A bis F)
if eqm_ventes(y, y) is None:
    print('Ergänzen Sie die Funktion eqm_ventes.')
else:
    a_mc = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
    b_mc = y.mean() - a_mc * x.mean()
    attendus = {'moindres carrés': ((a_mc, b_mc), 45.36, 93.29), 'ouverture': ((3, 30), 57.33, 134.25),
                'collègue': ((3.3, 14), 65.11, 70.73)}
    for nom, ((a, b), att_af, att_v) in attendus.items():
        e1, e2 = eqm_ventes(y, a * x + b), eqm_ventes(yv, a * xv + b)
        print(f'{nom:16s} A bis F: {e1:6.2f}   V1 bis V4: {e2:6.2f} →',
              'richtig' if abs(e1 - att_af) < 0.006 and abs(e2 - att_v) < 0.006 else 'bitte prüfen')
    r2_af = 1 - eqm_ventes(y, a_mc * x + b_mc) / eqm_ventes(y, np.full(6, y.mean()))
    print(f'R² auf A bis F = {r2_af:.4f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.4f} →',
          'richtig' if np.isclose(r2_af, np.corrcoef(x, y)[0, 1] ** 2) else 'bitte prüfen')

## Aufgabe 3.4 — Ein Abstieg mit standardisierten Flächen

*Daten D0.1. Begriffe: Gradient (Lektion 3.4), Standardisierung und Rückumrechnung (Lektion 3.5), Ausgleichsgerade der kleinsten Quadrate (Lektion 3.6).*

### a) Lösung

Mit $(w ; c) = (56{,}99 ; 190)$: $56{,}99 \times 1{,}4915 + 190 \approx 85{,}00 + 190 = 275{,}0$ k€. Mit $(a ; b) = (2{,}5 ; 30)$: $2{,}5 \times 98 + 30 = 245 + 30 = 275$ k€. Beide Schreibweisen ergeben dieselbe Vorhersage für F, wie Gleichung 2.7 ankündigt.

### b) Mittelwert, Standardabweichung und standardisierte Werte

In [ ]:
mu = x.mean()            # 64 m²
sigma = x.std()          # 22,796 m²: np.std teilt standardmäßig durch n
z = (x - mu) / sigma
z.round(4)

In [ ]:
# Überprüfung
if z is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('z-Werte:', 'richtig' if np.isclose(np.mean(z), 0) and np.isclose(np.mean(z ** 2), 1) and np.isclose(mu, 64) else 'bitte prüfen')

### c) Der Gradient in der standardisierten Schreibweise

In [ ]:
def gradient_std(w, c):
    e = y - (w * z + c)
    return np.array([-2 / len(x) * np.sum(z * e), -2 / len(x) * np.sum(e)])

In [ ]:
# Überprüfung am Startpunkt (gleiche Vorhersagen wie (2,5 ; 30))
if z is None or gradient_std(1, 1) is None:
    print('Ergänzen Sie die Zellen b) und c).')
else:
    w0, c0 = 2.5 * sigma, 30 + 2.5 * mu
    g0 = gradient_std(w0, c0)
    print(f'Start: w0 = {w0:.2f}, c0 = {c0:.0f}, gradient = {g0.round(2)} →',
          'richtig' if np.allclose(g0, [-29.683, -64.667], atol=0.001) else 'bitte prüfen')

### d) Dreißig Schritte, dann zurück zu den ursprünglichen Einheiten

In [ ]:
w, c = 2.5 * sigma, 30 + 2.5 * mu
for _ in range(30):
    w, c = np.array([w, c]) - 0.3 * gradient_std(w, c)
a_std = w / sigma
b_std = c - a_std * mu
print(f'w = {w:.3f}, c = {c:.3f}  →  a = {a_std:.3f} k€/m², b = {b_std:.2f} k€')

In [ ]:
# Überprüfung: Gleichung 3.9 und np.polyfit
a_mc = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
b_mc = y.mean() - a_mc * x.mean()
print(f'Gleichung 3.9: a = {a_mc:.3f}, b = {b_mc:.2f}; np.polyfit: {np.round(np.polyfit(x, y, 1), 3)}')
if a_std is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('erhaltene Gerade:', 'richtig' if np.allclose([a_std, b_std], [a_mc, b_mc], atol=1e-4) else 'bitte prüfen')

### e) und f) Der Verlust der beiden Abstiege, dann die Anzahl der Schritte

In [ ]:
# Die beiden Abstiege aus Lektion 3.5, Schritt für Schritt (nichts zu ergänzen)
def trajet_m2(nb_pas, eta=0.00006):
    a, b, pertes = 2.5, 30.0, []
    for _ in range(nb_pas + 1):
        pertes.append(np.mean((y - a * x - b) ** 2))
        e = y - a * x - b
        a, b = a + eta * 2 / 6 * np.sum(x * e), b + eta * 2 / 6 * np.sum(e)
    return pertes

def trajet_std(nb_pas, eta=0.3):
    zz = (x - x.mean()) / x.std()
    w, c, pertes = 2.5 * x.std(), 30 + 2.5 * x.mean(), []
    for _ in range(nb_pas + 1):
        pertes.append(np.mean((y - w * zz - c) ** 2))
        e = y - w * zz - c
        w, c = w + eta * 2 / 6 * np.sum(zz * e), c + eta * 2 / 6 * np.sum(e)
    return pertes

plt.figure(figsize=(6, 3))
plt.semilogy(trajet_m2(30), marker='o', color=ORANGE, label='Flächen in m², η = 0,000 06')
plt.semilogy(trajet_std(30), marker='s', color=BLEU, label='standardisierte Flächen, η = 0,3')
plt.axhline(45.364, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('Anzahl der Schritte'); plt.ylabel('Verlust (k€²)'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# f) Anzahl der Schritte, bis der Verlust weniger als 0,1 k€² über dem Minimum liegt (einige Sekunden)
L_min = np.mean((y - (a_mc * x + b_mc)) ** 2)

def nb_pas(X, a, b, eta, limite=10 ** 6):
    for k in range(1, limite + 1):
        e = y - a * X - b
        a, b = a + eta * 2 / 6 * np.sum(X * e), b + eta * 2 / 6 * np.sum(e)
        if np.mean((y - a * X - b) ** 2) - L_min < 0.1:
            return k

pertes_std = trajet_std(30)
print('standardisierte Flächen, η = 0,3:', next(k for k, v in enumerate(pertes_std) if v - L_min < 0.1), 'Schritte')
print('Flächen in m², η = 0,000 06:      ', nb_pas(x, 2.5, 30.0, 0.00006), 'Schritte')

### Lösung zu f)

Der standardisierte Abstieg kommt in **6 Schritten** auf weniger als 0,1 k€² an das Minimum heran, der Abstieg in m² in **169 808 Schritten**. Beide finden dieselbe Gerade, 3,151 × Fläche + 20,67. Merke: Vor einem Gradientenabstieg standardisiert man Spalten mit sehr unterschiedlichen Skalen. Um die Koeffizienten in den ursprünglichen Einheiten zu lesen, rechnet man sie anschließend mit Gleichung 3.8 zurück.

## Aufgabe 3.5 — Vorhersagen, dann überprüfen: Flächen in Einheiten von zehn m²

*Daten D0.1. Begriffe: Maßstabswechsel (Lektion 2.5), Gradient (Lektion 3.4), Standardisierung (Lektion 3.5).*

### a) bis d) Erwartete Vorhersagen

**a)** Die Gerade ändert sich nicht: Es sind dieselben Wohnungen und dieselben Preise, nur die Einheit der Spalte ändert sich. Eine Einheit von zehn m² entspricht 10 m², also wird die Steigung mit 10 multipliziert: 31,51 k€ pro zehn m². Der Achsenabschnitt, in k€, bleibt 20,67.

**b)** Die Starteinstellung wird zu (25 ; 30). Die Abweichungen bleiben gleich, und jedes $x_i$ wird durch 10 geteilt. Der Quotient beträgt also 74,5 / 10 ≈ 7,45 Einheiten von zehn m², das heißt weiterhin 74,5 m²: dieselbe Fläche, in einer anderen Einheit ausgedrückt.

**c)** Größer. Die Ableitung nach der Steigung wird durch 10 geteilt, und dieselbe Verschiebung der Geraden erfordert eine 10-mal größere Steigung. Die Grenz-Lernrate wird also mit etwa 10 × 10 = 100 multipliziert: Sie steigt von etwa 0,000 22 auf etwa 0,021.

**d)** Weniger Schritte als beim Abstieg in m², weil die Schritte 100-mal größer sein können. Aber mehr als beim standardisierten Abstieg: Die Flächen sind nicht zentriert, und das Tal bleibt geneigt.

### e) Überprüfen

In [ ]:
x_diz = x / 10
x_diz

In [ ]:
# Überprüfung, dann die Messungen (sonst nichts zu ergänzen)
if x_diz is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('x_diz:', 'richtig' if np.allclose(x_diz, [3.2, 4.5, 5.4, 7.0, 8.5, 9.8]) else 'bitte prüfen')
    print('a) Ausgleichsgerade (Steigung, Achsenabschnitt):', np.round(np.polyfit(x_diz, y, 1), 2))
    e0 = y - (25 * x_diz + 30)
    print(f'b) Quotient bei Schritt 0: {np.sum(x_diz * e0) / np.sum(e0):.2f} Einheiten von zehn m²')
    for eta in (0.021, 0.022):
        a, b = 25.0, 30.0
        for _ in range(200):
            e = y - a * x_diz - b
            a, b = a + eta * 2 / 6 * np.sum(x_diz * e), b + eta * 2 / 6 * np.sum(e)
        print(f'c) η = {eta}: Verlust nach 200 Schritten = {np.mean((y - a * x_diz - b) ** 2):.4g} k€²')
    L_min = np.mean((y - np.polyval(np.polyfit(x, y, 1), x)) ** 2)
    a, b = 25.0, 30.0
    for k in range(1, 10 ** 6):
        e = y - a * x_diz - b
        a, b = a + 0.006 * 2 / 6 * np.sum(x_diz * e), b + 0.006 * 2 / 6 * np.sum(e)
        if np.mean((y - a * x_diz - b) ** 2) - L_min < 0.1:
            break
    print(f'd) η = 0,006: {k} Schritte bis weniger als 0,1 k€² über dem Minimum')

### e) und f) Lösung

Das Notebook bestätigt die Vorhersagen: Steigung 31,51 k€ pro zehn m², Achsenabschnitt 20,67 k€; Quotient 7,45; mit 0,021 konvergiert der Abstieg, mit 0,022 divergiert er; mit $\eta = 0{,}006$ braucht es **1785 Schritte**, gegenüber 169 808 für die Flächen in m² und 6 für die standardisierten Flächen (Aufgabe 3.4).

Der Wechsel der Einheit hat die optimale Gerade nicht verändert: 31,51 k€ pro zehn m² sind 3,151 k€ pro m². Er hat den Weg und die nutzbare Lernrate verändert. Die Standardisierung geht weiter. Sie teilt durch die Standardabweichung und bringt so beide Koeffizienten auf dieselbe Skala. Sie zieht außerdem den Mittelwert ab: Mit $\sum z_i = 0$ behindern sich Steigung und Achsenabschnitt nicht mehr, und das Tal ist nicht mehr geneigt (Aufgabe 3.6).

## Aufgabe 3.6 — Herausforderung: Kreise auf der Karte

*Daten D0.1. Begriffe: Standardisierung (Lektionen 1.4 und 3.5), partielle Ableitungen und Gradient (Lektion 3.4).* Die Fragen a) bis f) werden auf Papier bearbeitet. Das Notebook überprüft Ihre Ergebnisse.

### a) bis f) Lösung

**a)** $\sum z_i = \sum (x_i - \mu)/\sigma = (\sum x_i - n\mu)/\sigma = 0$, da $n\mu = \sum x_i$. Und $\sum z_i^2 = \sum (x_i - \mu)^2 / \sigma^2 = n\sigma^2 / \sigma^2 = n$, da die Standardabweichung durch $n$ teilt.

**b)** $\partial L / \partial c = -\frac{2}{n} \sum (y_i - w z_i - c) = -\frac{2}{n}(n\bar{y} - w \times 0 - nc) = 2(c - \bar{y})$. $\partial L / \partial w = -\frac{2}{n} \sum z_i (y_i - w z_i - c) = -\frac{2}{n}(nm - wn - c \times 0) = 2(w - m)$. Beide Ableitungen verschwinden für $w^* = m$ und $c^* = \bar{y} = 222{,}33$ k€. Hier ist $m = 1\,637{,}5 / 22{,}796 \approx 71{,}83$ k€. Mit Gleichung 3.8 zurückgerechnet: $a = m / \sigma = \operatorname{cov}(x, y)/\sigma_x^2 \approx 3{,}151$, die Steigung aus Gleichung 3.9.

**c)** $w_{\text{nouveau}} = w - \eta \times 2(w - w^*)$, also $w_{\text{nouveau}} - w^* = (1 - 2\eta)(w - w^*)$. Für $c$ ist die Rechnung dieselbe. Mit $\eta = 0{,}3$ beträgt der Faktor $1 - 0{,}6 = 0{,}4$. In Aufgabe 3.2 war die Ableitung bereits $2(b - \bar{r})$, daher derselbe Faktor.

**d)** Mit $\eta = 0{,}5$ beträgt der Faktor 0: Ein einziger Schritt führt zum Minimum. Der Abstieg konvergiert, solange $|1 - 2\eta| < 1$ gilt, also $0 < \eta < 1$. Mit $\eta = 1$ beträgt der Faktor −1: Die Einstellung springt endlos zwischen zwei Punkten hin und her. Oberhalb von 1 divergiert der Abstieg.

**e)** Durch Ausmultiplizieren des Quadrats und mit a): $L = \overline{y^2} + w^2 + c^2 - 2wm - 2c\bar{y} = (w - m)^2 + (c - \bar{y})^2 + \overline{y^2} - m^2 - \bar{y}^2$, wobei $\overline{y^2}$ der Mittelwert der quadrierten Preise ist. Der letzte Term ist der Wert von $L$ im Minimum: $54\,637{,}33 - 5\,159{,}86 - 49\,432{,}11 \approx 45{,}36$ k€². Eine Höhenlinie fasst die Einstellungen mit gleichem Verlust zusammen, also mit gleichem Abstand $(w - w^*)^2 + (c - c^*)^2$ zum Minimum: Sie ist ein Kreis um das Minimum. Der Gradient zeigt zum Mittelpunkt, daher der fast geradlinige Weg in Lektion 3.5.

**f)** In m² gilt $\sum x_i = 384 \neq 0$ und $\frac{1}{n}\sum x_i^2 \approx 4\,615{,}67 \neq 1$. Die Entwicklung von $L(a, b)$ enthält dann einen gemischten Term $2 \times 64\, ab$, und $a^2$ wiegt etwa 4600-mal stärker als $b^2$. Die Höhenlinien sind stark gestreckte, geneigte Ellipsen: Das ist das Tal aus der Abbildung im Buch.

### Im Notebook überprüfen

In [ ]:
zz = (x - x.mean()) / x.std()
print(f'a) Summe der z = {np.sum(zz):.1e}, Summe der z² = {np.sum(zz ** 2):.4f}')

In [ ]:
m = np.mean(zz * y)

def un_pas(w, c, eta):
    """Gibt (w, c) nach einer Aktualisierung zurück (Gleichung 3.7 mit z statt x)."""
    e = y - (w * zz + c)
    return w + eta * 2 / len(x) * np.sum(zz * e), c + eta * 2 / len(x) * np.sum(e)

In [ ]:
# Überprüfung von b) und d)
if m is None or un_pas(1, 1, 0.1) is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print(f'b) m = {m:.2f} →', 'richtig' if np.isclose(m, np.mean((x - x.mean()) * (y - y.mean())) / x.std()) else 'bitte prüfen')
    w0, c0 = 2.5 * x.std(), 30 + 2.5 * x.mean()
    w1, c1 = un_pas(w0, c0, 0.5)
    print(f'd) ein Schritt mit η = 0,5: (w ; c) = ({w1:.2f} ; {c1:.2f}) →',
          'richtig' if np.allclose([w1, c1], [m, y.mean()]) else 'bitte prüfen')
    for eta in (1.0, 1.05):
        w, c = w0, c0
        suite = []
        for _ in range(4):
            w, c = un_pas(w, c, eta)
            suite.append(round(c, 2))
        print(f'   η = {eta}: c nach 1, 2, 3, 4 Schritten = {suite}')
    W, Cc = np.meshgrid(np.linspace(40, 100, 200), np.linspace(180, 260, 200))
    Z = np.mean([(yi - W * zi - Cc) ** 2 for zi, yi in zip(zz, y)], axis=0)
    plt.figure(figsize=(4.5, 4.5))
    plt.contour(W, Cc, Z, levels=[50, 100, 200, 400, 800, 1600], colors=BLEU)
    plt.gca().set_aspect('equal')
    plt.xlabel('w (k€ pro Standardabweichung)'); plt.ylabel('c (k€)'); plt.title('e) Kreise'); plt.show()